1. TAB638   - Population by municipality, age (5yr), sex, marital status (not using marital status) 
2. TAB6569  - Population by municipality, sex, swedish/other citizenship
3. TAB4320  - Education by municipality, age (10yr) 16-95+, sex, education level 

In [3]:
import requests
import pandas as pd
import numpy as np
from io import StringIO

/Users/shriya/opt/anaconda3/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/Users/shriya/opt/anaconda3/lib/python3.9/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.4' currently installed).
  from pandas.core import (


#### Mapping

In [2]:
MUNICIPALITIES = [
    "0114","0115","0117","0120","0123","0125","0126","0127","0128","0136","0138","0139",
    "0140","0160","0162","0163","0180","0181","0182","0183","0184","0186","0187","0188",
    "0191","0192","0305","0319","0330","0331","0360","0380","0381","0382","0428","0461",
    "0480","0481","0482","0483","0484","0486","0488","0509","0512","0513","0560","0561",
    "0562","0563","0580","0581","0582","0583","0584","0586","0604","0617","0642","0643",
    "0662","0665","0680","0682","0683","0684","0685","0686","0687","0760","0761","0763",
    "0764","0765","0767","0780","0781","0821","0834","0840","0860","0861","0862","0880",
    "0881","0882","0883","0884","0885","0980","1060","1080","1081","1082","1083","1214",
    "1230","1231","1233","1256","1257","1260","1261","1262","1263","1264","1265","1266",
    "1267","1270","1272","1273","1275","1276","1277","1278","1280","1281","1282","1283",
    "1284","1285","1286","1287","1290","1291","1292","1293","1315","1380","1381","1382",
    "1383","1384","1401","1402","1407","1415","1419","1421","1427","1430","1435","1438",
    "1439","1440","1441","1442","1443","1444","1445","1446","1447","1452","1460","1461",
    "1462","1463","1465","1466","1470","1471","1472","1473","1480","1481","1482","1484",
    "1485","1486","1487","1488","1489","1490","1491","1492","1493","1494","1495","1496",
    "1497","1498","1499","1715","1730","1737","1760","1761","1762","1763","1764","1765",
    "1766","1780","1781","1782","1783","1784","1785","1814","1860","1861","1862","1863",
    "1864","1880","1881","1882","1883","1884","1885","1904","1907","1960","1961","1962",
    "1980","1981","1982","1983","1984","2021","2023","2026","2029","2031","2034","2039",
    "2061","2062","2080","2081","2082","2083","2084","2085","2101","2104","2121","2132",
    "2161","2180","2181","2182","2183","2184","2260","2262","2280","2281","2282","2283",
    "2284","2303","2305","2309","2313","2321","2326","2361","2380","2401","2403","2404",
    "2409","2417","2418","2421","2422","2425","2460","2462","2463","2480","2481","2482",
    "2505","2506","2510","2513","2514","2518","2521","2523","2560","2580","2581","2582",
    "2583","2584"
]
MUN_STR = ",".join(MUNICIPALITIES)

In [3]:
MUN_NAMES = {
    114:"Upplands Väsby",115:"Vallentuna",117:"Österåker",120:"Värmdö",
    123:"Järfälla",125:"Ekerö",126:"Huddinge",127:"Botkyrka",128:"Salem",
    136:"Haninge",138:"Tyresö",139:"Upplands-Bro",140:"Nykvarn",160:"Täby",
    162:"Danderyd",163:"Sollentuna",180:"Stockholm",181:"Södertälje",
    182:"Nacka",183:"Sundbyberg",184:"Solna",186:"Lidingö",187:"Vaxholm",
    188:"Norrtälje",191:"Sigtuna",192:"Nynäshamn",305:"Håbo",319:"Älvkarleby",
    330:"Knivsta",331:"Heby",360:"Tierp",380:"Uppsala",381:"Enköping",
    382:"Östhammar",428:"Vingåker",461:"Gnesta",480:"Nyköping",481:"Oxelösund",
    482:"Flen",483:"Katrineholm",484:"Eskilstuna",486:"Strängnäs",488:"Trosa",
    509:"Ödeshög",512:"Ydre",513:"Kinda",560:"Boxholm",561:"Åtvidaberg",
    562:"Finspång",563:"Valdemarsvik",580:"Linköping",581:"Norrköping",
    582:"Söderköping",583:"Motala",584:"Vadstena",586:"Mjölby",604:"Aneby",
    617:"Gnosjö",642:"Mullsjö",643:"Habo",662:"Gislaved",665:"Vaggeryd",
    680:"Jönköping",682:"Nässjö",683:"Värnamo",684:"Sävsjö",685:"Vetlanda",
    686:"Eksjö",687:"Tranås",760:"Uppvidinge",761:"Lessebo",763:"Tingsryd",
    764:"Alvesta",765:"Älmhult",767:"Markaryd",780:"Växjö",781:"Ljungby",
    821:"Högsby",834:"Torsås",840:"Mörbylånga",860:"Hultsfred",861:"Mönsterås",
    862:"Emmaboda",880:"Kalmar",881:"Nybro",882:"Oskarshamn",883:"Västervik",
    884:"Vimmerby",885:"Borgholm",980:"Gotland",1060:"Olofström",
    1080:"Karlskrona",1081:"Ronneby",1082:"Karlshamn",1083:"Sölvesborg",
    1214:"Svalöv",1230:"Staffanstorp",1231:"Burlöv",1233:"Vellinge",
    1256:"Östra Göinge",1257:"Örkelljunga",1260:"Bjuv",1261:"Kävlinge",
    1262:"Lomma",1263:"Svedala",1264:"Skurup",1265:"Sjöbo",1266:"Hörby",
    1267:"Höör",1270:"Tomelilla",1272:"Bromölla",1273:"Osby",1275:"Perstorp",
    1276:"Klippan",1277:"Åstorp",1278:"Båstad",1280:"Malmö",1281:"Lund",
    1282:"Landskrona",1283:"Helsingborg",1284:"Höganäs",1285:"Eslöv",
    1286:"Ystad",1287:"Trelleborg",1290:"Kristianstad",1291:"Simrishamn",
    1292:"Ängelholm",1293:"Hässleholm",1315:"Hylte",1380:"Halmstad",
    1381:"Laholm",1382:"Falkenberg",1383:"Varberg",1384:"Kungsbacka",
    1401:"Härryda",1402:"Partille",1407:"Öckerö",1415:"Stenungsund",
    1419:"Tjörn",1421:"Orust",1427:"Sotenäs",1430:"Munkedal",1435:"Tanum",
    1438:"Dals-Ed",1439:"Färgelanda",1440:"Ale",1441:"Lerum",1442:"Vårgårda",
    1443:"Bollebygd",1444:"Grästorp",1445:"Essunga",1446:"Karlsborg",
    1447:"Gullspång",1452:"Åmål",1460:"Lilla Edet",1461:"Mellerud",
    1462:"Bengtsfors",1463:"Mariestad",1465:"Lidköping",1466:"Götene",
    1470:"Tibro",1471:"Töreboda",1472:"Vara",1473:"Skara",
    1480:"Göteborg",1481:"Mölndal",1482:"Kungälv",1484:"Lysekil",
    1485:"Uddevalla",1486:"Strömstad",1487:"Västra Götaland",1488:"Trollhättan",
    1489:"Alingsås",1490:"Borås",1491:"Ulricehamn",1492:"Tranemo",
    1493:"Bengtsfors",1494:"Lidköping",1495:"Skara",1496:"Skövde",
    1497:"Hjo",1498:"Tidaholm",1499:"Falköping",1715:"Kil",1730:"Eda",
    1737:"Torsby",1760:"Storfors",1761:"Hammarö",1762:"Munkfors",
    1763:"Forshaga",1764:"Grums",1765:"Årjäng",1766:"Sunne",1780:"Karlstad",
    1781:"Kristinehamn",1782:"Filipstad",1783:"Hagfors",1784:"Arvika",
    1785:"Säffle",1814:"Lekeberg",1860:"Laxå",1861:"Hallsberg",
    1862:"Degerfors",1863:"Hällefors",1864:"Ljusnarsberg",1880:"Örebro",
    1881:"Kumla",1882:"Askersund",1883:"Karlskoga",1884:"Nora",1885:"Lindesberg",
    1904:"Skinnskatteberg",1907:"Surahammar",1960:"Kungsör",1961:"Hallstahammar",
    1962:"Norberg",1980:"Västerås",1981:"Sala",1982:"Fagersta",1983:"Köping",
    1984:"Arboga",2021:"Vansbro",2023:"Malung-Sälen",2026:"Gagnef",
    2029:"Leksand",2031:"Rättvik",2034:"Orsa",2039:"Älvdalen",2061:"Smedjebacken",
    2062:"Mora",2080:"Falun",2081:"Borlänge",2082:"Säter",2083:"Hedemora",
    2084:"Avesta",2085:"Ludvika",2101:"Ockelbo",2104:"Hofors",2121:"Ovanåker",
    2132:"Nordanstig",2161:"Ljusdal",2180:"Gävle",2181:"Sandviken",
    2182:"Söderhamn",2183:"Bollnäs",2184:"Hudiksvall",2260:"Ånge",
    2262:"Timrå",2280:"Härnösand",2281:"Sundsvall",2282:"Kramfors",
    2283:"Sollefteå",2284:"Örnsköldsvik",2303:"Ragunda",2305:"Bräcke",
    2309:"Krokom",2313:"Strömsund",2321:"Åre",2326:"Berg",2361:"Härjedalen",
    2380:"Östersund",2401:"Nordmaling",2403:"Bjurholm",2404:"Vindeln",
    2409:"Robertsfors",2417:"Norsjö",2418:"Malå",2421:"Storuman",
    2422:"Sorsele",2425:"Dorotea",2460:"Vännäs",2462:"Vilhelmina",
    2463:"Åsele",2480:"Umeå",2481:"Lycksele",2482:"Skellefteå",
    2505:"Arvidsjaur",2506:"Arjeplog",2510:"Jokkmokk",2513:"Överkalix",
    2514:"Kalix",2518:"Kiruna",2521:"Gällivare",2523:"Pajala",
    2560:"Älvsbyn",2580:"Luleå",2581:"Piteå",2582:"Älvsbyn",
    2583:"Haparanda",2584:"Övertorneå"
}

In [4]:
# Education labels and mapping based on SNES labels (6 cut - V7302)

EDU_LABELS = {
    "1":  "Primary <9yr",
    "2":  "Primary 9-10yr",
    "3":  "Upper secondary <=2yr",   
    "4":  "Upper secondary 3yr",    
    "5":  "Post-secondary <3yr",   
    "6":  "University 3yr+",        
    "7":  "Postgraduate",
    "US": "Unknown",
}

EDU_TO_SNES6 = {
    "Primary <9yr":          "Primary <9yr",
    "Primary 9-10yr":        "Primary 9-10yr",
    "Upper secondary <=2yr": "Upper secondary",
    "Upper secondary 3yr":   "Upper secondary",
    "Post-secondary <3yr":   "Tertiary <2yr",
    "University 3yr+":       "Tertiary 2yr+",
    "Postgraduate":          "Postgraduate",
    "Unknown":               None,
}

In [5]:
# 5yr age group to 10yr group mapping 
# mapping based on age categories in snes (7 cut - V7202)

MAP_5YR_TO_EDU10 = {
    "18-19": "16-24",  "20-24": "16-24",
    "25-29": "25-34",  "30-34": "25-34",
    "35-39": "35-44",  "40-44": "35-44",
    "45-49": "45-54",  "50-54": "45-54",
    "55-59": "55-64",  "60-64": "55-64",
    "65-69": "65-74",  "70-74": "65-74",
    "75-79": "75-84",  "80-84": "75-84", 
    "85-89": "85-94",  "90-94": "85-94",  
    "95-99": "95+",    "100+":  "95+",   
}

AGE_TO_SNES7 = {
    "18-19": "18-22",
    "20-24": "18-22",
    "25-29": "23-30",
    "30-34": "31-40",
    "35-39": "31-40",
    "40-44": "41-50",
    "45-49": "41-50",
    "50-54": "51-60",
    "55-59": "51-60",
    "60-64": "61-70",
    "65-69": "61-70",
    "70-74": "71-104",
    "75-79": "71-104",
    "80-84": "71-104",
    "85-89": "71-104",
    "90-94": "71-104",
    "95-99": "71-104",
    "100+":  "71-104",
}

#### Recoding

In [6]:
def fetch_csv(url: str) -> pd.DataFrame:
    full = url if "outputFormat" in url else url + "&outputFormat=csv"
    r = requests.get(full, timeout=180)
    r.raise_for_status()
    df = pd.read_csv(StringIO(r.text), dtype=str)
    df.columns = [c.strip().lower() for c in df.columns]
    return df

def find_col(df: pd.DataFrame, *keywords) -> str:
    for col in df.columns:
        if any(k in col for k in keywords):
            return col

def recode_sex(s: pd.Series) -> pd.Series:
    return s.map({"1": "M", "2": "F", "men": "M", "women": "F",
                  "man": "M", "kvinna": "F"}).fillna(s)

def recode_age(s: pd.Series) -> pd.Series:
    return s.map(AGE_TO_SNES7).fillna(s)

def recode_education(s: pd.Series) -> pd.Series:
    return s.map(EDU_TO_SNES6)  

#### Getting the Tables

In [7]:
# Population Table

def download_population() -> pd.DataFrame:
  
    url = (
        "https://statistikdatabasen.scb.se/api/v2/tables/TAB638/data"
        "?lang=en"
        "&valueCodes[ContentsCode]=BE0101N1"
        f"&valueCodes[Region]={MUN_STR}"
        "&valueCodes[Civilstand]=OG,G,%C3%84NKL,SK"
        "&valueCodes[Alder]=-4,5-9,10-14,15-19,20-24,25-29,30-34,35-39,"
        "40-44,45-49,50-54,55-59,60-64,65-69,70-74,75-79,80-84,85-89,"
        "90-94,95-99,100%2B,us"
        "&valueCodes[Kon]=1,2"
        "&valueCodes[Tid]=2024"
        "&codelist[Region]=vs_RegionKommun07"
        "&codelist[Alder]=agg_%C3%85lder5%C3%A5r"
        "&outputValues[Alder]=aggregated"
    )
    df = fetch_csv(url)

    # keywords
    mun_col = find_col(df, "region")
    age_col = find_col(df, "ålder", "alder", "age")
    sex_col = find_col(df, "kön", "kon", "sex")
    
    skip = {"region", "age", "sex", "civil", "tid", "year", "content",
            "ålder", "alder", "kön", "kon"}
    val_col = next(c for c in df.columns if not any(k in c for k in skip))

    df = df.rename(columns={
        mun_col: "Municipality",
        age_col: "AgeRaw",
        sex_col: "Sex",
        val_col: "N"
    })

    df["N"] = pd.to_numeric(df["N"], errors="coerce").fillna(0)
    df["Sex"] = recode_sex(df["Sex"])

    # Sum over marital status (there will be 4 rows per municipality/age/sex combo) - collapsing marital status entirely
    df = df.groupby(["Municipality", "AgeRaw", "Sex"], as_index=False)["N"].sum()

    # Drop children/non-voters
    df = df[~df["AgeRaw"].isin(["-4", "0-4", "5-9", "10-14"])].copy()

    # Relabeling partial youngest group (label approximation)
    df["AgeGroup"] = df["AgeRaw"].replace({"15-19": "18-19"})

    return df[["Municipality", "AgeGroup", "Sex", "N"]].rename(columns={"N": "N_total"})

In [8]:
def download_citizenship() -> pd.DataFrame:
    url = (
        "https://statistikdatabasen.scb.se/api/v2/tables/TAB6569/data"
        "?lang=en"
        "&valueCodes[ContentsCode]=000007Y2"
        f"&valueCodes[Region]={MUN_STR}"
        "&valueCodes[Medborgarskap]=sv,SA"   
        "&valueCodes[Kon]=1,2"             
        "&valueCodes[Tid]=2024"
        "&codelist[Region]=vs_CRegionKommun07"
    )
    df = fetch_csv(url)

    mun_col  = find_col(df, "region")
    sex_col  = find_col(df, "kön", "kon", "sex")
    cit_col  = find_col(df, "medborgar", "citizen")
    skip     = {mun_col, sex_col, cit_col}
    val_col  = next(c for c in df.columns if c not in skip
                    and not any(k in c for k in ["tid", "year", "content"]))

    df = df.rename(columns={
        mun_col: "Municipality",
        sex_col: "Sex",
        cit_col: "CitCode",
        val_col: "N"
    })

    df["N"]   = pd.to_numeric(df["N"], errors="coerce").fillna(0)
    df["Sex"] = recode_sex(df["Sex"])

    swedish = (
        df[df["CitCode"].astype(str).str.strip() == "sv"]
        .rename(columns={"N": "N_swedish"})
        [["Municipality", "Sex", "N_swedish"]]
    )
    total = (
        df[df["CitCode"].astype(str).str.strip() == "SA"]
        .rename(columns={"N": "N_total_cit"})
        [["Municipality", "Sex", "N_total_cit"]]
    )

    out = total.merge(swedish, on=["Municipality", "Sex"], how="left")
    out["N_swedish"] = out["N_swedish"].fillna(0)
    out["citizen_share"] = (
        out["N_swedish"] / out["N_total_cit"].replace(0, np.nan)
    ).fillna(0.88)   #fallback - just in case

    return out[["Municipality", "Sex", "citizen_share"]]

In [9]:
def download_education() -> pd.DataFrame:
    url = (
        "https://statistikdatabasen.scb.se/api/v2/tables/TAB4320/data"
        "?lang=en"
        "&valueCodes[ContentsCode]=000000I2"
        f"&valueCodes[Region]={MUN_STR}"
        "&valueCodes[UtbildningsNiva]=1,2,3,4,5,6,7,US"
        "&valueCodes[Alder]=16-24,25-34,35-44,45-54,55-64,65-74,75-84,85-94,95%2B"
        "&valueCodes[Kon]=1,2"
        "&valueCodes[Tid]=2024"
        "&codelist[Region]=vs_RegionKommun07"
        "&codelist[Alder]=agg_%C3%85lder10%C3%A5r16-95%2B"
        "&outputValues[Alder]=aggregated"
    )
    df = fetch_csv(url)

    mun_col = find_col(df, "region")
    age_col = find_col(df, "ålder", "alder", "age")
    sex_col = find_col(df, "kön", "kon", "sex")
    edu_col = find_col(df, "utbildnings", "education", "niva", "level")
    skip    = {mun_col, age_col, sex_col, edu_col}
    val_col = next(c for c in df.columns if c not in skip
                   and not any(k in c for k in ["tid", "year", "content"]))

    df = df.rename(columns={
        mun_col: "Municipality",
        age_col: "AgeEdu",
        sex_col: "Sex",
        edu_col: "EduCode",
        val_col: "N_edu"
    })

    df["N_edu"] = pd.to_numeric(df["N_edu"], errors="coerce").fillna(0)
    df["Sex"]   = recode_sex(df["Sex"])
    df["Education"] = df["EduCode"].astype(str).str.strip().map(EDU_LABELS).fillna("Unknown")

    # education shares 
    totals = (
        df.groupby(["Municipality", "AgeEdu", "Sex"])["N_edu"]
        .sum().reset_index().rename(columns={"N_edu": "N_edu_total"})
    )
    df = df.merge(totals, on=["Municipality", "AgeEdu", "Sex"])
    df["edu_share"] = (
        df["N_edu"] / df["N_edu_total"].replace(0, np.nan)
    ).fillna(0)

    df["edu_share"] = (
    df["N_edu"] / df["N_edu_total"].replace(0, np.nan)
    ).fillna(0)

    return df[["Municipality", "AgeEdu", "Sex", "Education", "edu_share"]]

#### Building the Frame

In [10]:
def build_frame(pop: pd.DataFrame,
                cit: pd.DataFrame,
                edu: pd.DataFrame) -> pd.DataFrame:

    frame = pop.merge(cit, on=["Municipality", "Sex"], how="left")
    frame["N_citizens"] = (frame["N_total"] * frame["citizen_share"]).round(1)

    frame["AgeEdu"] = frame["AgeGroup"].map(MAP_5YR_TO_EDU10)
    frame = frame.merge(
        edu[["Municipality", "AgeEdu", "Sex", "Education", "edu_share"]],
        on=["Municipality", "AgeEdu", "Sex"],
        how="left"
    )
    frame["edu_share"] = frame["edu_share"].fillna(0)
    cell_edu_sum = frame.groupby(
        ["Municipality", "AgeGroup", "Sex"])["edu_share"].transform("sum")
    frame["edu_share"] = frame["edu_share"] / cell_edu_sum.replace(0, np.nan)

    frame["N"] = (frame["N_citizens"] * frame["edu_share"]).round(1)
    frame["Education"] = recode_education(frame["Education"])
    frame = frame.dropna(subset=["Education"])
    frame["Age"] = recode_age(frame["AgeGroup"])

    frame["MunicipalityCode"] = frame["Municipality"].astype(int).astype(str).str.zfill(4)
    frame["MunicipalityName"] = pd.to_numeric(
        frame["Municipality"], errors="coerce").map(MUN_NAMES)
    frame["Sex"] = frame["Sex"].map({"M": "Male", "F": "Female"}).fillna(frame["Sex"])

    frame = frame.groupby(
        ["MunicipalityCode", "MunicipalityName", "Age", "Sex", "Education"],
        as_index=False
    )["N"].sum()

    out = frame[frame["N"] > 0].copy()
    out["N"] = out["N"].round(1)
    out = out.sort_values(
        ["MunicipalityCode", "Age", "Sex", "Education"]).reset_index(drop=True)

    print(f"  ✓ {len(out):,} cells")
    print(f"     Municipalities : {out['MunicipalityCode'].nunique()}")
    print(f"     Age groups     : {sorted(out['Age'].unique())}")
    print(f"     Sex            : {sorted(out['Sex'].unique())}")
    print(f"     Education      : {sorted(out['Education'].unique())}")
    print(f"     Total N        : {out['N'].sum():,.0f}")
    return out

In [11]:
if __name__ == "__main__":
    pop = download_population()
    cit = download_citizenship()
    edu = download_education()

    frame = build_frame(pop, cit, edu)
    frame.to_csv("sweden_strat_frame.csv", index=False)

  ✓ 23,301 cells
     Municipalities : 290
     Age groups     : ['18-22', '23-30', '31-40', '41-50', '51-60', '61-70', '71-104']
     Sex            : ['Female', 'Male']
     Education      : ['Postgraduate', 'Primary 9-10yr', 'Primary <9yr', 'Tertiary 2yr+', 'Tertiary <2yr', 'Upper secondary']
     Total N        : 7,903,797


#### Adding Past Vote (trying)

In [27]:
frame = pd.read_csv("sweden_strat_frame.csv")
snes = pd.read_stata("VU2022_CSES.dta")
ri = pd.read_excel("ri_tab4_eng_2022.xlsx")

In [37]:

# municipality codes from electoral district codes
party_cols = ["M", "C", "L", "KD", "MP", "S", "V", "SD", "OTHER"]
all_mun_codes = frame["MunicipalityCode"].unique().tolist()
all_mun_codes_sorted = sorted(all_mun_codes, key=lambda x: len(str(int(x))), reverse=True)

def extract_mun_code(district_code):
    s = str(int(district_code))
    for mun in all_mun_codes_sorted:
        if s.startswith(str(int(mun))):
            return mun
    return None

ri["MunicipalityCode"] = ri["Electoral district code"].apply(extract_mun_code)

In [39]:
# Aggregating to municipality level for shares

mun_votes = (
    ri.groupby("MunicipalityCode")[party_cols + ["Number of valid ballot papers"]]
    .sum().reset_index()
)
for col in party_cols:
    mun_votes[f"share_2022_{col}"] = mun_votes[col] / mun_votes["Number of valid ballot papers"]

mun_votes = mun_votes.rename(columns={
    "share_2022_M": "share_2022_Moderate_Party",
    "share_2022_C": "share_2022_Centre_Party",
    "share_2022_L": "share_2022_The_Liberals",
    "share_2022_KD": "share_2022_Christian_Democrats",
    "share_2022_MP": "share_2022_Green_Party",
    "share_2022_S": "share_2022_Social_Democrats",
    "share_2022_V": "share_2022_Left_Party",
    "share_2022_SD": "share_2022_Sweden_Democrats",
    "share_2022_OTHER": "share_2022_Other",
})

share_cols = [c for c in mun_votes.columns if c.startswith("share_2022_")]

# Merging onto the frame
frame_augmented = frame.merge(
    mun_votes[["MunicipalityCode"] + share_cols],
    on="MunicipalityCode", how="left"
)

In [40]:
frame_augmented.head()

,MunicipalityCode,MunicipalityName,Age,Sex,Education,N,share_2022_Moderate_Party,share_2022_Centre_Party,share_2022_The_Liberals,share_2022_Christian_Democrats,share_2022_Green_Party,share_2022_Social_Democrats,share_2022_Left_Party,share_2022_Sweden_Democrats,share_2022_Other
0,114,Upplands Väsby,18-22,Female,Primary 9-10yr,944.2,0.205188,0.058939,0.043384,0.049869,0.036205,0.301413,0.076231,0.209356,0.019415
1,114,Upplands Väsby,18-22,Female,Primary <9yr,6.8,0.205188,0.058939,0.043384,0.049869,0.036205,0.301413,0.076231,0.209356,0.019415
2,114,Upplands Väsby,18-22,Female,Tertiary 2yr+,80.8,0.205188,0.058939,0.043384,0.049869,0.036205,0.301413,0.076231,0.209356,0.019415
3,114,Upplands Väsby,18-22,Female,Tertiary <2yr,263.5,0.205188,0.058939,0.043384,0.049869,0.036205,0.301413,0.076231,0.209356,0.019415
4,114,Upplands Väsby,18-22,Female,Upper secondary,927.7,0.205188,0.058939,0.043384,0.049869,0.036205,0.301413,0.076231,0.209356,0.019415
